# 📊 Day 02 — Missing Values Deep Dive
## Project: Loan Default Predictor

---
**Goal today:** Understand WHY values are missing, visualize the pattern, and DECIDE (but not yet fully implement) the imputation strategy.

**Time estimate:** 1.5 – 2 hours

**What you will do:**
1. Install & use `missingno` for visual missingness patterns
2. Understand types of missingness (MCAR / MAR / MNAR)
3. Check if missing income → higher default rate
4. Decide imputation strategy per column
5. Implement basic imputation
6. Verify imputation worked correctly

> 💡 **Rule of the Day:** Never impute blindly. Always ask WHY the value is missing first. That WHY changes your strategy.

---
## SECTION 0 — Theory You Must Know Before Coding

### The 3 Types of Missingness (Know this for interviews!)

| Type | Full Name | Meaning | Example |
|---|---|---|---|
| **MCAR** | Missing Completely At Random | No pattern. Random data entry errors. | A form field accidentally skipped |
| **MAR** | Missing At Random | Missingness depends on OTHER columns | Older people less likely to report income |
| **MNAR** | Missing Not At Random | Missingness depends on the missing value itself | High earners hide their income |

> ⚠️ **MNAR is the most dangerous.** If high-risk borrowers hide their income, then imputing with median income gives those people a FAKE safe-looking value. The model then underestimates risk for exactly the people it should flag.

### Imputation Strategies
| Strategy | When to Use |
|---|---|
| **Mean imputation** | Normal distribution, few missing values |
| **Median imputation** | Skewed distribution (like income) — more robust |
| **Mode imputation** | Categorical columns |
| **Fill with 0** | When missing = truly 0 (e.g. no dependents) |
| **Flag + impute** | When missingness itself is informative — add a binary `_was_missing` column |

---
## SECTION 1 — Setup

> ❇️ **What is rcParams** --> rcParams stands for Runtime Configuration Parameters. This changes the plotting size for any future plot automatically

In [4]:
# Install missingno if not already installed
# Run this only once — comment it out after
# !pip install missingno

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5) # rcParams stands for Runtime Configuration Parameters
sns.set_style('whitegrid')

print('✅ Libraries ready')

✅ Libraries ready


In [5]:
# Load the raw data (always work from raw, never from a previously modified version)
df = pd.read_csv('cs-training.csv', index_col=0)
df_original = df.copy()  # Keep a backup — ALWAYS do this before any changes

print(f'✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

✅ Loaded: 150,000 rows × 11 columns


---
## SECTION 2 — Visualize Missingness with `missingno`

`missingno` gives us 3 powerful charts to understand missing data patterns:
- **Matrix** → shows which rows have missing values side by side
- **Bar** → shows completeness per column
- **Heatmap** → shows if two columns tend to be missing together (correlation)

In [ ]:
# Matrix plot — white lines = missing values
# If white lines ALIGN across columns → missingness is correlated (not random)
print('=== MISSING VALUES MATRIX ===')
print('White lines = missing values. Look for vertical alignment patterns.')
msno.matrix(df, figsize=(14, 6), fontsize=10, color=(0.2, 0.4, 0.8))
plt.title('Missing Values Matrix — White = Missing', fontweight='bold')
plt.tight_layout()
plt.savefig('msno_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart — shows % completeness per column
msno.bar(df, figsize=(14, 5), fontsize=10, color='steelblue')
plt.title('Column Completeness (Bar Chart)', fontweight='bold')
plt.tight_layout()
plt.savefig('msno_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap — do columns go missing TOGETHER?
# High positive correlation = when col A is missing, col B is also missing
msno.heatmap(df, figsize=(10, 6), fontsize=10)
plt.title('Missingness Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig('msno_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### ✍️ YOUR OBSERVATIONS — Missingness Patterns

- From the matrix: Do the white lines align? → YES / NO
- From the heatmap: Are MonthlyIncome and NumberOfDependents missing together? → ___
- My guess for missingness type:
  - `MonthlyIncome`: likely **MAR or MNAR** because _(your reasoning)_
  - `NumberOfDependents`: likely **MAR** because _(your reasoning)_

---
## SECTION 3 — KEY ANALYSIS: Does Missing Income = Higher Default?

This is the most important analysis of Day 2.

**Hypothesis:** If people who hide their income are riskier borrowers, then the missing income group should have a HIGHER default rate.

This determines our imputation strategy:
- If YES → we must add a `MonthlyIncome_was_missing` flag column (the missingness is informative!)
- If NO → simple median imputation is fine

In [ ]:
target = 'SeriousDlqin2yrs'

# Create a flag for missing income
df['income_missing_flag'] = df['MonthlyIncome'].isnull().astype(int)

# Compare default rates
default_by_missing = df.groupby('income_missing_flag')[target].agg(['mean', 'count'])
default_by_missing.index = ['Income Provided', 'Income Missing']
default_by_missing.columns = ['Default Rate', 'Count']
default_by_missing['Default Rate %'] = (default_by_missing['Default Rate'] * 100).round(2)

print('=== DEFAULT RATE BY INCOME AVAILABILITY ===')
print(default_by_missing[['Count', 'Default Rate %']])

In [ ]:
# Visualize the difference
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(default_by_missing.index,
              default_by_missing['Default Rate %'],
              color=colors, edgecolor='white', linewidth=2, width=0.5)

for bar, val in zip(bars, default_by_missing['Default Rate %']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val}%', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_ylabel('Default Rate (%)')
ax.set_title('Does Missing Income Predict Default?\n(This determines our imputation strategy)',
             fontweight='bold')
ax.set_ylim(0, max(default_by_missing['Default Rate %']) * 1.3)
plt.tight_layout()
plt.savefig('missing_income_vs_default.png', dpi=150, bbox_inches='tight')
plt.show()

# Drop the temp flag column for now (we'll add it back properly later)
df.drop('income_missing_flag', axis=1, inplace=True)

### ✍️ YOUR OBSERVATIONS — Critical Finding

- Default rate with income provided: ___%
- Default rate with income missing: ___%
- Difference: ___% 

**Conclusion:**
- If difference > 2%: Missingness IS informative → We MUST add a `MonthlyIncome_was_missing` binary flag column before imputing
- If difference < 2%: Simple median imputation is fine

> 🎯 **Interview line:** *"I found that borrowers with missing income had a ___% higher default rate. This suggests missing income is not random — it's informative. So I added a binary flag column to preserve this signal before imputing with median."*

---
## SECTION 4 — Imputation Decision Table

Before writing a single line of imputation code, we document our decisions.
This is what separates thoughtful data scientists from people who just run code.

In [ ]:
# Document imputation decisions as a DataFrame (professional practice)
decisions = pd.DataFrame({
    'Column': ['MonthlyIncome', 'NumberOfDependents'],
    'Missing %': [
        round(df['MonthlyIncome'].isnull().mean() * 100, 2),
        round(df['NumberOfDependents'].isnull().mean() * 100, 2)
    ],
    'Missingness Type': ['MAR/MNAR', 'MAR'],
    'Strategy': [
        'Add flag column + Median imputation (stratified by age group)',
        'Fill with 0 (missing likely means no dependents)'
    ],
    'Reasoning': [
        'Income is right-skewed → median is robust. Flag preserves signal.',
        'No dependents is a valid state; blank = 0 is reasonable assumption.'
    ]
})

print('=== IMPUTATION STRATEGY DECISIONS ===')
print(decisions.to_string(index=False))

---
## SECTION 5 — Implement Imputation

Now we implement the strategy we just documented. Order matters:
1. Add flag columns FIRST (before imputing — otherwise you lose the missingness info)
2. Then impute

In [ ]:
# Step 1: Add flag column BEFORE imputing MonthlyIncome
df['MonthlyIncome_was_missing'] = df['MonthlyIncome'].isnull().astype(int)

print(f'Flag column added.')
print(f'Rows flagged as missing income: {df["MonthlyIncome_was_missing"].sum():,}')

In [ ]:
# Step 2: Stratified median imputation for MonthlyIncome
# Why stratified? Because a 25-year-old's typical income ≠ a 55-year-old's
# Stratifying by age group gives more realistic imputed values

# Create age groups
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 30, 40, 50, 60, 150],
                          labels=['<30', '30-40', '40-50', '50-60', '60+'])

# Compute median income per age group
median_by_age = df.groupby('age_group')['MonthlyIncome'].median()
print('=== MEDIAN INCOME BY AGE GROUP ===')
for group, val in median_by_age.items():
    print(f'  Age {group}: ${val:,.0f}/month')

In [ ]:
# Apply stratified imputation
def impute_income(row):
    if pd.isnull(row['MonthlyIncome']):
        return median_by_age.get(row['age_group'], df['MonthlyIncome'].median())
    return row['MonthlyIncome']

df['MonthlyIncome'] = df.apply(impute_income, axis=1)

# Drop the temporary age_group column (we'll re-create it properly in feature engineering)
df.drop('age_group', axis=1, inplace=True)

print(f'Missing income after imputation: {df["MonthlyIncome"].isnull().sum()}')

In [ ]:
# Step 3: Fill NumberOfDependents with 0
# Reasoning: Missing dependents almost certainly means 0 dependents
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

print(f'Missing dependents after fill: {df["NumberOfDependents"].isnull().sum()}')

---
## SECTION 6 — Verify Imputation Worked

In [ ]:
# Final missing value check — should show 0 for treated columns
print('=== MISSING VALUES AFTER IMPUTATION ===')
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else '✅ No missing values remaining!')

# Confirm new column exists
print(f'\nNew column added: MonthlyIncome_was_missing')
print(f'Value counts: {df["MonthlyIncome_was_missing"].value_counts().to_dict()}')

In [ ]:
# Compare income distribution before vs after imputation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cap = df_original['MonthlyIncome'].quantile(0.99)

# Before
df_original['MonthlyIncome'].dropna().clip(upper=cap).hist(
    bins=50, ax=ax1, color='#e74c3c', edgecolor='white', alpha=0.8)
ax1.set_title('MonthlyIncome BEFORE Imputation\n(missing = excluded)', fontweight='bold')
ax1.set_xlabel('Income ($)')

# After
df['MonthlyIncome'].clip(upper=cap).hist(
    bins=50, ax=ax2, color='#2ecc71', edgecolor='white', alpha=0.8)
ax2.set_title('MonthlyIncome AFTER Imputation\n(all rows included)', fontweight='bold')
ax2.set_xlabel('Income ($)')

plt.suptitle('Income Distribution: Before vs After Imputation', fontweight='bold')
plt.tight_layout()
plt.savefig('income_imputation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## SECTION 7 — Save Cleaned Data

In [ ]:
# Save the imputed dataset for use in Day 3+
df.to_csv('cs-training-day2-cleaned.csv', index=True)

print('=== DAY 2 COMPLETE ===')
print(f'Rows  : {df.shape[0]:,}')
print(f'Cols  : {df.shape[1]}  (added MonthlyIncome_was_missing)')
print(f'Missing values remaining: {df.isnull().sum().sum()}')
print()
print('FILES SAVED:')
print('  ✅ cs-training-day2-cleaned.csv')
print('  ✅ msno_matrix.png')
print('  ✅ msno_bar.png')
print('  ✅ msno_heatmap.png')
print('  ✅ missing_income_vs_default.png')
print('  ✅ income_imputation_comparison.png')
print()
print('TOMORROW — Day 3: Outlier Detection')
print('  → IQR method on all numeric columns')
print('  → Box plots for every feature')
print('  → Decide: cap, remove, or keep each outlier')

### ✍️ YOUR PERSONAL NOTES

**What I learned today:**
*(write here)*

**The most important decision I made and why:**
*(write here)*

**Questions I still have:**
*(write here)*

---
> ✅ **Day 2 Complete.** You didn't just fill nulls — you understood WHY they existed and made a defensible decision. That's the difference between a data scientist and someone who runs code.